# 087 — Modelos de difusión

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**DDPM** (Ho et al., 2020): el proceso **forward** es una cadena de Markov fija
que corrompe el dato con ruido gaussiano según un scheduler β₁,…,β_T:

```text
q(x_t | x_{t−1}) = N( √(1−β_t) · x_{t−1} ,  β_t · I )
```

Con α_t = 1 − β_t y ᾱ_t = ∏ α_s existe **forma cerrada** para saltar a cualquier t:

```text
x_t = √ᾱ_t · x₀ + √(1−ᾱ_t) · ε ,   ε ~ N(0, I)
```

El proceso **reverse** aprendido p_θ(x_{t−1}|x_t) = N(μ_θ(x_t, t), σ_t²I) invierte
la cadena partiendo de ruido puro. La red predice el ruido ε y se entrena con el
**objetivo simplificado** `L_simple = E[‖ε − ε_θ(x_t, t)‖²]` — una regresión
estable, sin juego adversarial. Muestrear cuesta T evaluaciones de red (DDIM lo
reduce a 20-100 sin reentrenar).

## 🧮 Ejemplo de referencia

Scheduler β₁ = 0.1, β₂ = 0.2 → α₁ = 0.9, α₂ = 0.8, **ᾱ₂ = 0.72**.
Con x₀ = 1.0 y ε = 0.5:

```text
x₂ = √0.72 · 1.0 + √0.28 · 0.5 = 0.8485 + 0.2646 = 1.1131
```

Si la red acierta el ruido (ε̂ = 0.5), despejar recupera el dato exacto:
x̂₀ = (1.1131 − 0.2646)/0.8485 = **1.0000**. Si predice ε̂ = 0.3, la pérdida es
(0.5 − 0.3)² = **0.04** y x̂₀ = 1.1247. Verifícalo a mano antes de ejecutar el
laboratorio.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("generation", seed=87)
show(result)


## Reflexión

1. ¿Por qué la forma cerrada x_t = √ᾱ_t·x₀ + √(1−ᾱ_t)·ε es imprescindible para entrenar
   con pasos t aleatorios sin simular la cadena completa?
2. El objetivo ‖ε − ε_θ(x_t, t)‖² no menciona verosimilitud: ¿qué relación guarda con
   el ELBO del VAE (clase 085) y por qué se considera una cota simplificada?
3. La difusión necesita decenas o cientos de pasadas de red para una muestra frente a
   una sola de GAN/VAE: ¿qué compra ese costo (estabilidad, cobertura de modos) y qué
   técnicas lo mitigan (DDIM, destilación)?
